In [ ]:
import os
import warnings
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Suppress minor compatibility warnings
warnings.filterwarnings("ignore", message="Core Pydantic V1 functionality")

# --- 1. Schemas for Structured Output ---

class LCMSMethod(BaseModel):
    """The hardware and chromatography settings proposed by the Generator."""
    column_type: str = Field(description="e.g., C4, Diphenyl, SEC")
    mobile_phase_a: str = Field(description="Aqueous phase composition")
    mobile_phase_b: str = Field(description="Organic phase composition")
    cone_voltage: int = Field(description="Cone voltage in V. Critical for intact proteins.")
    desolvation_temp: int = Field(description="Desolvation temperature in Celsius.")
    capillary_voltage: float = Field(description="Capillary voltage in kV.")
    reasoning: str = Field(description="Why these settings were chosen.")

class ReviewCritique(BaseModel):
    """The feedback from the Reflection Agent."""
    is_acceptable: bool = Field(description="True if the method is safe and optimal to run.")
    critique: str = Field(description="Specific feedback on physics or hardware limits.")

# --- 2. Updated Graph State (Includes Raw Sequences) ---

class AgentState(TypedDict):
    heavy_chain_seq: str
    light_chain_seq: str
    bsab_context: str       # Populated automatically by the extraction node
    method: LCMSMethod      # The current proposed method
    critique: str           # The latest feedback from the reflector
    is_acceptable: bool     # Pass/fail flag
    iterations: int         # Safety counter

# --- 3. Initialize Local Model ---
llm = ChatOllama(
    model="llama3:8b",      # Or your preferred local model
    temperature=0.1,
    num_ctx=16384,
    num_predict=1024
)

# --- 4. Define the Agent Nodes ---

def feature_extraction_node(state: AgentState) -> dict:
    """Deterministically extracts biophysical features from sequences using Biopython."""
    heavy_seq = state["heavy_chain_seq"]
    light_seq = state["light_chain_seq"]
    full_seq = heavy_seq + light_seq
    
    # Deterministic calculations
    analyzed_seq = ProteinAnalysis(full_seq)
    mol_weight = analyzed_seq.molecular_weight()
    isoelectric_point = analyzed_seq.isoelectric_point()
    
    # N-linked glycosylation sequon finder (N-X-[S/T], X != P)
    sequons = []
    for i in range(len(heavy_seq) - 2):
        trip = heavy_seq[i:i+3]
        if trip[0] == 'N' and trip[2] in ['S', 'T'] and trip[1] != 'P':
            sequons.append(i + 1)
            
    extracted_profile = f"""
    Molecule: IgG-like Bispecific Antibody
    - Calculated Intact Molecular Weight: {mol_weight:.2f} Da
    - Theoretical Isoelectric Point (pI): {isoelectric_point:.2f}
    - Detected N-Glycosylation Sequons (Heavy Chain positions): {sequons}
    - Sequence Lengths: Heavy ({len(heavy_seq)} aa), Light ({len(light_seq)} aa)
    """
    
    print("\n[FEATURE EXTRACTION NODE] Biophysical properties computed programmatically.")
    return {"bsab_context": extracted_profile}

def generator_node(state: AgentState) -> dict:
    """Proposes or refines the LC-MS method based on the extracted biophysical context."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert LC-MS method developer for Waters BioAccord/Select Series instruments.
        Your goal is to propose chromatography and MS source settings for intact bispecific antibody analysis based on the extracted biophysical properties.
        If you receive a critique, adjust your parameters to fix the physical or chemical errors."""),
        ("user", """
        Extracted Antibody Context: {bsab_context}
        
        Previous Critique to address: {critique}
        
        Generate the optimal LC-MS method parameters as JSON matching the requested schema.
        """)
    ])
    
    generator = prompt | llm.with_structured_output(LCMSMethod)
    critique = state.get("critique", "None - Initial Generation")
    
    result = generator.invoke({
        "bsab_context": state["bsab_context"],
        "critique": critique
    })
    
    return {"method": result, "iterations": state.get("iterations", 0) + 1}

def reflector_node(state: AgentState) -> dict:
    """Critiques the proposed method against biophysics and hardware constraints."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a senior wet-lab analytical chemist. Review the proposed Waters LC-MS method.
        Check strictly for these physical constraints for intact glycoprotein analysis:
        1. Cone voltage: >100V risks in-source fragmentation and shearing of labile sialic acids.
        2. Column type: C18 is for peptides. Intact proteins require C4, C8, or Diphenyl columns to avoid irreversible binding.
        3. Mobile phase: Must use volatile acids (e.g., 0.1% Formic Acid). Non-volatile salts will crash the mass spec.
        
        If any of these are violated, set is_acceptable to False and provide a harsh, precise critique."""),
        ("user", "Proposed Method:\n{method}")
    ])
    
    reflector = prompt | llm.with_structured_output(ReviewCritique)
    result = reflector.invoke({"method": state["method"].model_dump_json()})
    
    return {"critique": result.critique, "is_acceptable": result.is_acceptable}

# --- 5. Define Control Flow ---

def routing_condition(state: AgentState) -> Literal["generate", "__end__"]:
    if state["is_acceptable"]:
        print("\n[ROUTER] Method approved by Reflector. Ending debate.")
        return "__end__"
    elif state["iterations"] >= 3:
        print("\n[ROUTER] Max iterations reached. Forcing exit.")
        return "__end__"
    else:
        print("\n[ROUTER] Method rejected. Routing back to Generator for revision.")
        return "generate"

# Build the LangGraph workflow
workflow = StateGraph(AgentState)

workflow.add_node("extract_features", feature_extraction_node)
workflow.add_node("generate", generator_node)
workflow.add_node("reflect", reflector_node)

# Wiring the graph execution order
workflow.set_entry_point("extract_features")
workflow.add_edge("extract_features", "generate")
workflow.add_edge("generate", "reflect")
workflow.add_conditional_edges("reflect", routing_condition)

app = workflow.compile()

# --- 6. Execution with Raw Sequences ---

initial_state = {
    "heavy_chain_seq": "EVQLLESGGGLVQPGGSLRLSCAASGFTFNTYAMNWVRQAPGKGLEWVGRIRSKYNNYATYYADSVKDRFTISRDDSKNTAYLQMNSLKTEDTAVYYCARHGNFGNSYVSWFAYWGQGTLVTVSS",
    "light_chain_seq": "QAVVTQESALTTSPGETVTLTCRSSTGAVTTSNYANWVQEKPDHLFTGLIGGTNKRAPGVPARFSGSLIGDKAALTITGAQTEDEAIYFCALWYSNLWVFGGGTKLTVL",
    "iterations": 0,
    "is_acceptable": False,
    "critique": "",
    "bsab_context": ""
}

print("Starting Feature-Extraction & Agentic Debate Pipeline...")
for output in app.stream(initial_state):
    if "extract_features" in output:
        print(output["extract_features"]["bsab_context"])
    if "generate" in output:
        print("\n--- GENERATOR PROPOSED METHOD ---")
        print(output["generate"]["method"].model_dump_json(indent=2))
    if "reflect" in output:
        print("\n--- REFLECTOR CRITIQUE ---")
        print(f"Acceptable: {output['reflect']['is_acceptable']}")
        print(f"Critique: {output['reflect']['critique']}")

Starting Feature-Extraction & Agentic Debate Pipeline...

[FEATURE EXTRACTION NODE] Biophysical properties computed programmatically.

    Molecule: IgG-like Bispecific Antibody
    - Calculated Intact Molecular Weight: 25409.07 Da
    - Theoretical Isoelectric Point (pI): 8.34
    - Detected N-Glycosylation Sequons (Heavy Chain positions): []
    - Sequence Lengths: Heavy (125 aa), Light (109 aa)
    

--- GENERATOR PROPOSED METHOD ---
{
  "column_type": "ACQUITY UHPLC BEH C18, 1.7um, 2.1x100mm",
  "mobile_phase_a": "water with 0.1% formic acid",
  "mobile_phase_b": "acetonitrile with 0.1% formic acid",
  "cone_voltage": 30,
  "desolvation_temp": 150,
  "capillary_voltage": 3.5,
  "reasoning": "Based on the extracted biophysical properties, I propose the following LC-MS method parameters for the analysis of the intact bispecific antibody.\n\nThe calculated molecular weight of 25409.07 Da suggests a relatively large molecule, which requires a suitable chromatography column and MS sourc